<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/09_custom_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 独自埋め込み関数 (FAQ × sentence-transformers)

このノートブックは **pytidb シリーズの第 9 回** です。組み込みの TiDB Cloud 埋め込みではなく、ローカルで動く `sentence-transformers` を使った **独自 `BaseEmbeddingFunction`** を実装します。

## 学習目標
- `BaseEmbeddingFunction` を継承してクラスを書く
- 必須 3 メソッド (`get_query_embedding` / `get_source_embedding` / `get_source_embeddings`) を実装
- クライアントサイド埋め込み (`use_server=False`) を VectorField に渡す
- 小型の多言語モデル `intfloat/multilingual-e5-small` (約 120MB) で FAQ 検索を動かす

前提: `03_vector_search.ipynb` を実行済み。
API キーは不要。ただし初回のみモデルダウンロードに数十秒かかります。


## 1. 依存パッケージのインストール

今回は `sentence-transformers` を追加インストールします (PyTorch は Colab 既定で入っています)。


In [ ]:
!pip install -q pytidb sentence-transformers


## 2. TiDB Cloud Zero の払い出し


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-custom-embed")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 3. 独自の BaseEmbeddingFunction を作る

`sentence-transformers` の多言語小型モデルをラップします。
モデル: `intfloat/multilingual-e5-small` (384 次元)。日本語を含む 100 以上の言語に対応。


In [ ]:
from typing import Any, List, Optional
from pytidb.embeddings.base import BaseEmbeddingFunction
from sentence_transformers import SentenceTransformer


class E5SmallEmbedding(BaseEmbeddingFunction):
    # pydantic v2 ベースなので内部で持つ HF モデルは PrivateAttr 相当にしないと serialize されない
    # ここでは単純化のため object 型としてアトリビュートに付ける
    model_config = {"arbitrary_types_allowed": True}

    def __init__(self, model_name: str = "intfloat/multilingual-e5-small", **kwargs):
        # e5 系のモデルは 384 次元。provider は分類用の任意文字列で OK
        super().__init__(
            provider="sentence-transformers",
            model_name=model_name,
            dimensions=384,
            use_server=False,   # クライアント側で埋め込む
            **kwargs,
        )
        self.__dict__["_model"] = SentenceTransformer(model_name)

    def _encode(self, texts: List[str]) -> List[List[float]]:
        # e5 系モデルでは "query:"/"passage:" プリフィクスを付けると検索精度が上がる
        # ここでは利用者から見て透過的に扱いたいので、シンプルに encode のみを行う
        arr = self.__dict__["_model"].encode(texts, normalize_embeddings=True)
        return [row.tolist() for row in arr]

    def get_query_embedding(self, query: Any, source_type: Optional[str] = "text", **kwargs) -> List[float]:
        return self._encode([f"query: {query}"])[0]

    def get_source_embedding(self, source: Any, source_type: Optional[str] = "text", **kwargs) -> List[float]:
        return self._encode([f"passage: {source}"])[0]

    def get_source_embeddings(self, sources: List[Any], source_type: Optional[str] = "text", **kwargs) -> List[List[float]]:
        return self._encode([f"passage: {s}" for s in sources])


embed_fn = E5SmallEmbedding()
print("Embedding dim =", embed_fn.dimensions)
print("sample =", len(embed_fn.get_query_embedding("こんにちは、世界")), "次元ベクトル")


## 4. FAQ テーブルを作成 + データ投入

`question` 列を埋め込み対象にします。`embed_fn.VectorField(source_field="question")` を使うと、自動で投入時に埋め込みが作られます。


In [ ]:
from pytidb import TiDBClient
from pytidb.datatype import TEXT
from pytidb.schema import Field, TableModel

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database="faq_demo",
    ensure_db=True,
)


class FAQ(TableModel):
    __tablename__ = "faqs"
    __table_args__ = {"extend_existing": True}

    id: int = Field(primary_key=True)
    question: str = Field(sa_type=TEXT)
    answer: str = Field(sa_type=TEXT)
    question_vec: list[float] = embed_fn.VectorField(source_field="question")


table = db.create_table(schema=FAQ, if_exists="overwrite")
print("テーブル準備OK:", table.table_name)


In [ ]:
FAQS = [
    ("注文のキャンセルはいつまでできますか?",
     "発送準備前であればマイページの注文履歴からキャンセルできます。"),
    ("送料はいくらですか?",
     "通常便は全国一律 550 円、5,000 円以上のお買い上げで送料無料です。"),
    ("海外発送はしていますか?",
     "現在は日本国内のみの発送となっております。"),
    ("領収書はもらえますか?",
     "マイページの注文詳細から PDF でダウンロードいただけます。"),
    ("商品が届かないときは?",
     "発送後 5 営業日を過ぎても届かない場合はカスタマーサポートまでお問い合わせください。"),
    ("返品はできますか?",
     "未使用・未開封の商品に限り、到着後 7 日以内であれば返品を受け付けます。"),
    ("支払い方法は何が使えますか?",
     "クレジットカード、コンビニ決済、銀行振込、後払い (Paidy) に対応しています。"),
    ("ポイントの有効期限は?",
     "最後にポイントを取得または利用した日から 1 年間有効です。"),
    ("会員登録は無料ですか?",
     "はい。無料で登録でき、登録時に 500 ポイントをプレゼントしています。"),
    ("パスワードを忘れた場合は?",
     "ログイン画面の「パスワードを忘れた方」から再設定用のメールをお送りします。"),
    ("メルマガを解除したい",
     "メール下部の購読解除リンク、またはマイページの通知設定から停止できます。"),
    ("プレゼント用にラッピングできますか?",
     "一部商品を除き、200 円でギフトラッピングを承ります。注文時に選択してください。"),
    ("届いた商品が壊れていた場合は?",
     "到着後 3 日以内にカスタマーサポートへご連絡ください。無償で交換いたします。"),
    ("注文後に住所を変更したい",
     "発送準備前であればマイページから変更できます。発送後は配送業者までご連絡ください。"),
    ("店舗でも購入できますか?",
     "オンライン限定商品は店舗では扱っておりません。その他の商品は各店舗でお買い求めいただけます。"),
]

table.bulk_insert([FAQ(id=i + 1, question=q, answer=a) for i, (q, a) in enumerate(FAQS)])
print(f"投入完了: {table.rows()} 件")


## 5. 独自埋め込みで検索する

`table.search(QUERY)` の中では、`embed_fn.get_query_embedding(QUERY)` が呼ばれて
生成された 384 次元のベクトルで検索が行われます。


In [ ]:
queries = [
    "キャンセルしたい",
    "送料のことが知りたい",
    "paypayで払えますか?",
    "商品が動かなくて困っています",
    "どれくらいで届きますか?",
]
for q in queries:
    print(f"\n--- Q: {q}")
    for r in table.search(q).limit(3).to_pydantic():
        print(f"  sim={r.similarity_score:.3f}  Q:{r.hit.question}")
        print(f"    A:{r.hit.answer}")


## 追加実験

- `intfloat/multilingual-e5-base` (768 次元) に差し替えて精度と速度を比べる
- `normalize_embeddings=False` にしてスコア挙動がどう変わるか観察
- `sparse_vecs` を追加するなど独自のマルチベクトル戦略を実装する
